# 🔄 Data Preprocessing Pipeline
## Cross-Platform User Matching

Pipeline นี้แบ่งการทำงานออกเป็น **6 Stage** ที่แยกจากกันอย่างชัดเจน:

| Stage | ชื่อ | หน้าที่ |
|-------|------|--------|
| 1 | **Configuration & Imports** | ตั้งค่า path, import libraries |
| 2 | **Data Loading** | โหลด JSON profiles จาก Dataset-LinkSocial |
| 3 | **Data Cleaning & Standardization** | NaN handling, normalize text, remove emojis |
| 4 | **Platform Splitting** | แยก DataFrame ตาม platform |
| 5 | **Ground Truth & Training Data** | สร้าง positive/negative pairs สำหรับ training |
| 6 | **Export & Quality Report** | บันทึกไฟล์ + สรุปคุณภาพข้อมูล |

---

## Stage 1: Configuration & Imports
ตั้งค่า path, import libraries ที่จำเป็น

In [ ]:
import os
import json
import re
import ast
import pandas as pd
import numpy as np
import unicodedata
import emoji
import math
import html
from ftfy import fix_text
from typing import Dict, Tuple
from urllib.parse import urlparse
from datetime import datetime


# =================================================
# ===========
# CONFIG — แก้ path ตรงนี้ที่เดียว
# ============================================================
BASE_PATH   = "../data/data/Dataset-LinkSocial"  # โฟลเดอร์ dataset ต้นทาง
OUTPUT_DIR  = "../data"                 # โฟลเดอร์ output

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config loaded")
print(f"   BASE_PATH  = {os.path.abspath(BASE_PATH)}")
print(f"   OUTPUT_DIR = {os.path.abspath(OUTPUT_DIR)}")

✅ Config loaded
   BASE_PATH  = d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data\data\Dataset-LinkSocial
   OUTPUT_DIR = d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data


---
## Stage 2: Data Loading
โหลด JSON profiles จากทั้ง 3 โฟลเดอร์ (`1.profile.data`, `2.profile.data`, `3.profile.data`)  
แต่ละไฟล์จะถูกระบุ platform จากชื่อไฟล์ (twitter / instagram / googleplus)

Step 2.1: **Load JSON Files** — โหลดไฟล์ JSON ทั้งหมด

In [9]:
def load_all_profiles(base_path: str) -> pd.DataFrame:
    """
    โหลด profiles ทั้งหมดจาก 3 โฟลเดอร์ profile.data

    Returns:
        pd.DataFrame ที่มี column: userName, fullName, bio, location,
        externalUrl, pictureURL, platform, source_folder, user_folder
    """
    all_profiles = []
    profile_folders = ["1.profile.data", "2.profile.data", "3.profile.data"]

    for folder in profile_folders:
        folder_path = os.path.join(base_path, folder)

        if not os.path.exists(folder_path):
            print(f"⚠️  Folder {folder_path} not found, skipping...")
            continue

        print(f"📂 Loading from {folder}...")

        # รวบรวม user_paths ทั้งหมด
        # 2.profile.data มีโครงสร้าง 3 ชั้น: folder/pair_folder/user_folder/
        # 1 & 3 มีโครงสร้าง 2 ชั้น: folder/user_folder/
        user_paths = []

        for entry in os.listdir(folder_path):
            entry_path = os.path.join(folder_path, entry)
            if not os.path.isdir(entry_path):
                continue

            # ตรวจสอบว่าเป็น pair folder (เช่น Google_Insta) หรือ user folder
            sub_entries = os.listdir(entry_path)
            has_json = any(f.endswith('.json') for f in sub_entries)

            if has_json:
                # โครงสร้าง 2 ชั้น: entry คือ user_folder ที่มี JSON อยู่ข้างใน
                user_paths.append((entry, entry_path))
            else:
                # โครงสร้าง 3 ชั้น: entry คือ pair folder → ต้องวนลูปอีกชั้น
                for user_folder in sub_entries:
                    user_path = os.path.join(entry_path, user_folder)
                    if os.path.isdir(user_path):
                        user_paths.append((user_folder, user_path))

        print(f"   Found {len(user_paths)} user folders")

        for user_folder_name, user_path in user_paths:
            for file in os.listdir(user_path):
                if file.endswith('.json'):
                    # ข้าม metadata files ที่ไม่ใช่ profile
                    if file in ('filename.json', 'score_file.json'):
                        continue

                    file_path = os.path.join(user_path, file)
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                            data['source_folder'] = folder
                            data['user_folder'] = user_folder_name

                            # ระบุ platform จากชื่อไฟล์
                            if 'twitter' in file.lower():
                                data['platform'] = 'twitter'
                            elif 'instagram' in file.lower():
                                data['platform'] = 'instagram'
                            elif 'google' in file.lower():
                                data['platform'] = 'googleplus'
                            else:
                                data['platform'] = 'unknown'

                            all_profiles.append(data)
                    except Exception as e:
                        print(f"❌ Error reading {file_path}: {e}")

    df = pd.DataFrame(all_profiles)

    print(f"\n{'='*50}")
    print(f"📦 Step 2.1 Summary")
    print(f"{'='*50}")
    print(f"Total JSON files loaded: {len(df)}")

    return df

# === โหลดไฟล์ทั้งหมด ===
df_raw = load_all_profiles(BASE_PATH)

📂 Loading from 1.profile.data...
   Found 1530 user folders


KeyboardInterrupt: 

Step 2.2: **Create DataFrame & Display** — สร้าง DataFrame และแสดงผล

In [8]:
# === แสดงผล DataFrame ===
print(f"{'='*50}")
print(f"📊 Step 2.2: DataFrame Summary")
print(f"{'='*50}")
print(f"Total profiles : {len(df_raw)}")
print(f"Total columns  : {len(df_raw.columns)}")
print(f"Columns        : {list(df_raw.columns)}")

print(f"\n📈 Platform distribution:")

print(df_raw['platform'].value_counts().to_string())

print(f"\n📂 Source folder distribution:")
print(df_raw['source_folder'].value_counts().to_string())

print(f"\n📋 DataFrame Info:")
df_raw.info()

print(f"\n🔍 Sample data (first 5 rows):")
df_raw.head()


📊 Step 2.2: DataFrame Summary


NameError: name 'df_raw' is not defined

Step 2.3 : **Save Raw CSV** — บันทึกเป็น CSV

In [ ]:
# === แสดงผล DataFrame ===
print(f"{'='*50}")
print(f"📊 Step 2.2: DataFrame Summary")
print(f"{'='*50}")
print(f"Total profiles : {len(df_raw)}")
print(f"Total columns  : {len(df_raw.columns)}")
print(f"Columns        : {list(df_raw.columns)}")

print(f"\n📈 Platform distribution:")
print(df_raw['platform'].value_counts().to_string())

print(f"\n📂 Source folder distribution:")
print(df_raw['source_folder'].value_counts().to_string())

print(f"\n📋 DataFrame Info:")
df_raw.info()

print(f"\n🔍 Sample data (first 5 rows):")
df_raw.head()


📊 Step 2.2: DataFrame Summary
Total profiles : 36807
Total columns  : 11
Columns        : ['userName', 'fullName', 'bigrams', 'source_folder', 'user_folder', 'platform', 'bio', 'externalUrl', 'outputProfileName', 'pictureURL', 'location']

📈 Platform distribution:
platform
twitter       13960
googleplus    11890
instagram     10957

📂 Source folder distribution:
source_folder
3.profile.data    23187
2.profile.data    12078
1.profile.data     1542

📋 DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 36807 entries, 0 to 36806
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   userName           36807 non-null  str   
 1   fullName           36423 non-null  str   
 2   bigrams            36807 non-null  object
 3   source_folder      36807 non-null  str   
 4   user_folder        36807 non-null  str   
 5   platform           36807 non-null  str   
 6   bio                31225 non-null  str   
 7   exte

,userName,fullName,bigrams,source_folder,user_folder,platform,bio,externalUrl,outputProfileName,pictureURL,location
0,i3mawi,Adeeb Amawi,"[i3, 3m, ma, aw, wi, Ad, de, ee, eb, b, A, Am,...",1.profile.data,A3mawi,googleplus,NaN,NaN,NaN,NaN,NaN
1,WoltersKluwerEspaa,Wolters Kluwer Espaa,"[Wo, ol, lt, te, er, rs, sK, Kl, lu, uw, we, e...",1.profile.data,A3Software,googleplus,NaN,NaN,NaN,NaN,NaN
2,AALISHANMATRIX,AALISHAN MATRIX,"[AA, AL, LI, IS, SH, HA, AN, NM, MA, AT, TR, R...",1.profile.data,aalishanmatrix,googleplus,"Computer scientist, Aspiring entrepreneur & in...",[http://www.siliconindia.com/profiles/aalishan...,NaN,NaN,NaN
3,@aaronbird,Aaron Bird,"[@a, aa, ar, ro, on, nb, bi, ir, rd, Aa, ar, r...",1.profile.data,aaronbird,twitter,"Co-Founder & Full Stack CEO @bizible, the B2B ...",bizible.com,aaronbird,https://pbs.twimg.com/profile_images/378800000...,"Queen Anne, Seattle, Cascadia"
4,acfoto,acfoto,"[ac, cf, fo, ot, to, ac, cf, fo, ot, to]",1.profile.data,AaronCohenArts,instagram,NaN,http://www.aaroncodling.com,AaronCohenArts,https://scontent-ord1-1.cdninstagram.com/t51.2...,NaN


Step 2.4 : **Verify Saved CSV** — ตรวจสอบ CSV ที่บันทึก

1. ทำการบันทึกช้อมูลลงในไฟล์ CSV

In [ ]:
output_dir = os.path.join(BASE_PATH, "data") 
os.makedirs(output_dir, exist_ok=True) 

raw_csv_path = os.path.join(output_dir, "combined_profiles.csv")

# 1. ฟังก์ชันสำหรับเซฟไฟล์ (เพิ่ม quoting เพื่อล็อกข้อความที่มีการขึ้นบรรทัดใหม่ไม่ให้ CSV พัง)
def save_full_data():
    print(f"\n💾 กำลังบันทึกไฟล์ใหม่ทั้งหมดไปที่: {raw_csv_path}...")
    df_raw.to_csv(raw_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

need_to_save = True

# 2. ตรวจสอบไฟล์เดิมว่ามีอยู่และข้อมูลครบไหม
if os.path.exists(raw_csv_path):
    print("🔍 พบไฟล์เดิมอยู่ กำลังตรวจสอบจำนวนข้อมูล...")
    try:
        # ลองโหลดไฟล์เดิมมานับแถว
        df_existing = pd.read_csv(raw_csv_path, low_memory=False)
        if len(df_existing) == len(df_raw):
            print(f"✅ ข้อมูลในไฟล์ CSV มีจำนวนแถวตรงกับข้อมูลดิบเป๊ะ ({len(df_raw)} แถว) ไม่ต้องบันทึกใหม่")
            need_to_save = False
        else:
            print(f"⚠️ ข้อมูลไม่ตรงกัน! (ในไฟล์มี {len(df_existing)} แถว, แต่ข้อมูลใหม่มี {len(df_raw)} แถว)")
            print("🔄 กำลังเตรียมเขียนทับใหม่ให้สมบูรณ์...")
    except Exception as e:
        print(f"⚠️ ไฟล์เดิมอ่านไม่ได้หรือพัง ({e}) กำลังเตรียมสร้างใหม่...")

# 3. สั่งเซฟและตรวจสอบความเรียบร้อย
if need_to_save:
    save_full_data()
    
    print("🔍 กำลังตรวจสอบความถูกต้องหลังบันทึก...")
    # โหลดไฟล์ที่เพิ่งเซฟเสร็จมาเช็ค (โหลดมาทั้งหมดเพื่อความชัวร์เรื่องจำนวนแถว)
    df_verify = pd.read_csv(raw_csv_path, low_memory=False)
    
    # ด่านตรวจ: จำนวนคอลัมน์และจำนวนแถวต้องเป๊ะ!
    assert len(df_verify.columns) == len(df_raw.columns), "❌ Error: จำนวนคอลัมน์หลังจากบันทึกไม่ตรง!"
    assert len(df_verify) == len(df_raw), f"❌ Error: จำนวนแถวหลังเซฟ ({len(df_verify)}) ยังไม่ตรงกับต้นฉบับ ({len(df_raw)})!"
    
    print(f"✅ บันทึกไฟล์สำเร็จ! ข้อมูลครบถ้วนทั้ง {len(df_verify)} แถวเรียบร้อยแล้ว!")

print(raw_csv_path)

🔍 พบไฟล์เดิมอยู่ กำลังตรวจสอบจำนวนข้อมูล...
✅ ข้อมูลในไฟล์ CSV มีจำนวนแถวตรงกับข้อมูลดิบเป๊ะ (36807 แถว) ไม่ต้องบันทึกใหม่
../data/data/Dataset-LinkSocial\data\combined_profiles.csv


Step 2.5 : ทำการเรียกใช้ Data จากไฟล์ csv ที่มีอยู่แล้ว

In [10]:

# ถอยหลัง 1 ก้าว (../) ออกจาก Train-Data แล้วเข้าโฟลเดอร์ data
file_path = "../data/data/Dataset-LinkSocial\data\combined_profiles.csv" 
print(file_path)
df = pd.read_csv(file_path)
print(df.shape)

../data/data/Dataset-LinkSocial\data\combined_profiles.csv
(36807, 11)


---
## Stage 3: Data Cleaning

### Step 3.1: ทำการ clean data เพื่อเอาไปใช้งานในขั้นตอนถัดไปโดยใช้ข้อมูลที่ copy มาจาก df_raw เพื่อป้องกันการที่เขียนทับข้อมูลจนข้อมูลดิบหาย

In [26]:
df_clean = df.copy()


### Step 3.2: NaN Handling
จัดการค่า **NaN / missing values** ใน text fields ทั้งหมด  
- Text columns (`userName`, `fullName`, `bio`, `location`, `externalUrl`, `pictureURL`) → แทนด้วย `''`  
- `outputProfileName` → ถ้าเป็น NaN ใช้ `user_folder` แทน

In [27]:

# --- ก่อนทำ: ดู NaN ทั้งหมด ---
text_columns = ['userName', 'fullName', 'bio', 'location', 'externalUrl', 'pictureURL']

print("📊 NaN Count — ก่อนทำ (Before):")
print("-" * 40)
for col in text_columns:
    if col in df_clean.columns:
        nan_count = df_clean[col].isna().sum()
        total = len(df_clean)
        pct = nan_count / total * 100
        print(f"  {col:15s}: {nan_count:>5} NaN ({pct:5.1f}%)")
    else:
        print(f"  {col:15s}: column ไม่มีใน DataFrame")

📊 NaN Count — ก่อนทำ (Before):
----------------------------------------
  userName       :    83 NaN (  0.2%)
  fullName       :   426 NaN (  1.2%)
  bio            :  6427 NaN ( 17.5%)
  location       : 23936 NaN ( 65.0%)
  externalUrl    :  6227 NaN ( 16.9%)
  pictureURL     : 12108 NaN ( 32.9%)


In [28]:
# --- ทำ NaN Handling ---

# Text fields: NaN → empty string
for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('')

# outputProfileName: ถ้าไม่มีให้ใช้ user_folder แทน
if 'outputProfileName' in df_clean.columns:
    df_clean['outputProfileName'] = df_clean['outputProfileName'].fillna(df_clean['user_folder'])
else:
    df_clean['outputProfileName'] = df_clean['user_folder']

# --- หลังทำ: ดู NaN อีกที ---
print("📊 NaN Count — หลังทำ (After):")
print("-" * 40)
for col in text_columns:
    if col in df_clean.columns:
        nan_count = df_clean[col].isna().sum()
        print(f"  {col:15s}: {nan_count:>5} NaN")

# ดูจำนวน empty string ด้วย
print(f"\n📊 Empty String Count (หลัง fillna):")
print("-" * 40)
for col in text_columns:
    if col in df_clean.columns:
        empty_count = (df_clean[col] == '').sum()
        total = len(df_clean)
        pct = empty_count / total * 100
        print(f"  {col:15s}: {empty_count:>5} empty ({pct:5.1f}%)")

print(f"\n✅ Step 3.2 NaN Handling เสร็จ")

📊 NaN Count — หลังทำ (After):
----------------------------------------
  userName       :     0 NaN
  fullName       :     0 NaN
  bio            :     0 NaN
  location       :     0 NaN
  externalUrl    :     0 NaN
  pictureURL     :     0 NaN

📊 Empty String Count (หลัง fillna):
----------------------------------------
  userName       :    83 empty (  0.2%)
  fullName       :   426 empty (  1.2%)
  bio            :  6427 empty ( 17.5%)
  location       : 23936 empty ( 65.0%)
  externalUrl    :  6227 empty ( 16.9%)
  pictureURL     : 12108 empty ( 32.9%)

✅ Step 3.2 NaN Handling เสร็จ


In [29]:
# 1. กำหนดรายชื่อคอลัมน์ที่ต้องการ (ตามที่คุณเลือกมา 6 คอลัมน์)
cols_to_combine = ['userName', 'fullName', 'bio', 'location', 'externalUrl', 'pictureURL']

# 2. สร้าง DataFrame ใหม่โดยเลือกเฉพาะคอลัมน์เหล่านี้
# ใช้ .copy() เพื่อป้องกัน Warning เวลาเราไปแก้ไขข้อมูลในตัวแปรใหม่นี้
df_already = df_clean[cols_to_combine].copy()

df_already.head()


,userName,fullName,bio,location,externalUrl,pictureURL
0,i3mawi,Adeeb Amawi,,,,
1,WoltersKluwerEspaa,Wolters Kluwer Espaa,,,,
2,AALISHANMATRIX,AALISHAN MATRIX,"Computer scientist, Aspiring entrepreneur & in...",,['http://www.siliconindia.com/profiles/aalisha...,
3,@aaronbird,Aaron Bird,"Co-Founder & Full Stack CEO @bizible, the B2B ...","Queen Anne, Seattle, Cascadia",bizible.com,https://pbs.twimg.com/profile_images/378800000...
4,acfoto,acfoto,,,http://www.aaroncodling.com,https://scontent-ord1-1.cdninstagram.com/t51.2...


ทำการเก็บบันทึกข้อมูลลงไปในไฟล์ csv ตาม path ที่กำหนดเอาไว้ 

In [30]:

output_path = r'Project-for-Work\data\data\Dataset-LinkSocial\data\cleaned_social_data.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_already.to_csv(output_path, index=False, encoding='utf-8-sig')

In [31]:
df_already.count()

userName       36807
fullName       36807
bio            36807
location       36807
externalUrl    36807
pictureURL     36807
dtype: int64

### Step 3.3: Normalization ของแต่ละ คอลลัมน์


In [32]:
df_nomalized = df_already.copy()

### Step 3.3: Clean Username
`userName` → `userName_clean`

การแปลง:
- trim → lowercase → ลบ `@` นำหน้า → ลบ emoji
- ลบ special chars (เก็บ `a-z 0-9 . _ -`)
- ยุบตัวคั่นซ้ำ (`..` → `.`, `__` → `_`, `--` → `-`)
- ลบตัวคั่นหัว/ท้าย
- ตัวอย่าง: `@A_A.ron🔥` → `a_a.ron`


In [36]:
def remove_emojis(text: str) -> str:
    if pd.isna(text) or not isinstance(text, str):
        return ""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

In [37]:

def clean_username(text: str) -> str:
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = re.sub(r'^@+', '', text)
    text = remove_emojis(text)
    text = re.sub(r'[^a-z0-9._\-]', '', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'_{2,}', '_', text)
    text = re.sub(r'-{2,}', '-', text)
    text = text.strip('._-')
    return text

# Cell 19 (Step 3.3)
df_nomalized.loc[:, 'userName'] = df_nomalized['userName'].apply(clean_username)



In [40]:
df_nomalized.head(5)

,userName,fullName,bio,location,externalUrl,pictureURL
0,i3mawi,Adeeb Amawi,,,,
1,wolterskluwerespaa,Wolters Kluwer Espaa,,,,
2,aalishanmatrix,AALISHAN MATRIX,"Computer scientist, Aspiring entrepreneur & in...",,['http://www.siliconindia.com/profiles/aalisha...,
3,aaronbird,Aaron Bird,"Co-Founder & Full Stack CEO @bizible, the B2B ...","Queen Anne, Seattle, Cascadia",bizible.com,https://pbs.twimg.com/profile_images/378800000...
4,acfoto,acfoto,,,http://www.aaroncodling.com,https://scontent-ord1-1.cdninstagram.com/t51.2...


แสดงผลข้อมูลที่มีการเปลี่ยนแปลง

In [39]:
comparison_df = pd.DataFrame()

comparison_df['Original_username'] = df_already['userName']
comparison_df['Cleaned_username'] = df_nomalized['userName']


comparison_df.head(30)

,Original_username,Cleaned_username
0,i3mawi,i3mawi
1,WoltersKluwerEspaa,wolterskluwerespaa
2,AALISHANMATRIX,aalishanmatrix
3,@aaronbird,aaronbird
4,acfoto,acfoto
5,GuantanamoBae,guantanamobae
6,@aaronzlewis,aaronzlewis
7,iconicguy,iconicguy
8,AlexBindaFernndez,alexbindafernndez
9,AbelSerral,abelserral


### Step 3.4: Clean FullName
`fullName` → `fullName_clean`

การแปลง:
- trim → lowercase → ลบ emoji
- ลบคำนำหน้า (`Mr.`, `Mrs.`, `Ms.`, `Dr.`, `Prof.`)
- ลบ special chars (เก็บ `a-z 0-9 space`)
- ยุบช่องว่างซ้ำ → เหลือช่องเดียว
- ตัวอย่าง: `Mr. Aaron   Bird ✨` → `aaron bird`


In [41]:
def clean_fullname(text: str) -> str:
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = remove_emojis(text)
    text = re.sub(r'\b(mr|mrs|ms|dr|prof)\.?\s*', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

# Cell 19 (Step 3.3)
df_nomalized.loc[:, 'fullName'] = df_nomalized['fullName'].apply(clean_fullname)

In [42]:
df_nomalized 

,userName,fullName,bio,location,externalUrl,pictureURL
0,i3mawi,adeeb amawi,,,,
1,wolterskluwerespaa,wolters kluwer espaa,,,,
2,aalishanmatrix,aalishan matrix,"Computer scientist, Aspiring entrepreneur & in...",,['http://www.siliconindia.com/profiles/aalisha...,
3,aaronbird,aaron bird,"Co-Founder & Full Stack CEO @bizible, the B2B ...","Queen Anne, Seattle, Cascadia",bizible.com,https://pbs.twimg.com/profile_images/378800000...
4,acfoto,acfoto,,,http://www.aaroncodling.com,https://scontent-ord1-1.cdninstagram.com/t51.2...
...,...,...,...,...,...,...
36802,zuzugraphics,mind in your brand,"Graphic Designer in San Francisco - Wings, Hea...",,http://www.diego-t.com/index4.html,https://scontent-ord1-1.cdninstagram.com/t51.2...
36803,zuzugraphics,diego taborda,"I am a Digital Artist, freelance web/Graphic D...",San Francisco,about.me/zuzugraphics,https://pbs.twimg.com/profile_images/857844253...
36804,tomruneberg,tom rune berg,Everything is relative ;) NITH Skedsmo Videreg...,,"['http://picasaweb.google.com/zwirwel', 'http:...",
36805,zwirwel,tom berg,,,,https://scontent-ord1-1.cdninstagram.com/t51.2...


In [44]:
comparison_df = pd.DataFrame()

comparison_df['Original_fullName'] = df_already['fullName']
comparison_df['Cleaned_fullName'] = df_nomalized['fullName']
comparison_df.head()

,Original_fullName,Cleaned_fullName
0,Adeeb Amawi,adeeb amawi
1,Wolters Kluwer Espaa,wolters kluwer espaa
2,AALISHAN MATRIX,aalishan matrix
3,Aaron Bird,aaron bird
4,acfoto,acfoto


### Step 3.5: Clean Bio 
`bio` → `bio_clean`

การแปลง:
1. แปลงค่าว่างเป็น "" และ trim หน้า-หลัง

2. ยุบช่องว่างหลายอัน/ขึ้นบรรทัดใหม่ ให้เหลือ space เดียว

3. lowercase ทั้งข้อความ

4, แยก URL ออกมาในอีก column

5. แยก @mention ออก มาเมื่อมีการ @ ให้แยกไว้ว่ามีกี่ @ แล้วก็เก็บโดยใช้ | คั้นเอาไว้

6, hashtag ให้ลบแค่ # แต่เก็บคำไว้ เช่น #DataScience → datascience

7. emoji ไม่ลบทิ้ง แต่แปลงเป็นข้อความ เช่น 🚀 → rocket เพราะมีงานวิจัยที่พบว่าการแทน emoji ด้วยคำอธิบายช่วยคงสัญญาณความหมายได้ดีกว่าการลบทิ้งในหลายงาน NLP

8. ลบตัวคั่นรก ๆ ที่ไม่ช่วยความหมาย เช่น |, •, >>> แต่ไม่ต้องกวาด punctuation ทุกตัวแบบแรงเกินไป

9. ยุบช่องว่างอีกรอบ แล้วจบ

In [46]:

# === Regex patterns (compile ครั้งเดียว) ===
URL_RE = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
MENTION_RE = re.compile(r'@(\w+)')
HASHTAG_RE = re.compile(r'#(\w+)')
NOISY_SEP_RE = re.compile(r'[|•·►▸▹▻>>>★☆✦✧~≡]{1,}')
MULTISPACE_RE = re.compile(r'\s+')
def safe_str(value) -> str:
    """Convert any value to safe string"""
    if value is None:
        return ''
    if isinstance(value, float) and math.isnan(value):
        return ''
    s = str(value).strip()
    if s.lower() == 'none':
        return ''
    return s
def extract_urls(text: str) -> list:
    """ดึง URL ทั้งหมดออกจาก text"""
    return URL_RE.findall(text)
def extract_mentions(text: str) -> list:
    """ดึง @mention ทั้งหมดออกจาก text (ไม่รวม @)"""
    return MENTION_RE.findall(text)
def clean_single_url(url: str) -> str:
    """Clean URL: ลบ protocol, www., trailing /"""
    text = url.strip().lower()
    text = re.sub(r'^https?://', '', text)
    text = re.sub(r'^www\.', '', text)
    text = text.rstrip('/')
    return text
def clean_bio(text) -> str:
    """ทำความสะอาด bio text — ได้ข้อความสะอาดกลับมา"""
    # 1. แปลงค่าว่างเป็น "" และ trim
    text = safe_str(text)
    if not text:
        return ''
    # 2. แก้ broken unicode (ftfy)
    text = fix_text(text)
    # 3. แปลง HTML entities (&amp; → &, &lt; → <)
    text = html.unescape(text)
    # 4. ยุบช่องว่าง/newline หลายอัน → space เดียว
    text = MULTISPACE_RE.sub(' ', text).strip()
    # 5. lowercase
    text = text.lower()
    # 6. ลบ URL (แยกเก็บใน column อื่น)
    text = URL_RE.sub(' ', text)
    # 7. ลบ @mention (แยกเก็บใน column อื่น)
    text = MENTION_RE.sub(' ', text)
    # 8. #hashtag → ลบ # เก็บคำ
    text = HASHTAG_RE.sub(r'\1', text)
    # 9. emoji → ข้อความ (🚀 → rocket)
    text = emoji.demojize(text, delimiters=(' ', ' '))
    # 10. ลบตัวคั่นรก ๆ
    text = NOISY_SEP_RE.sub(' ', text)
    # 11. ยุบช่องว่างอีกรอบ + trim
    text = MULTISPACE_RE.sub(' ', text).strip()
    return text if len(text) >= 3 else ''

# Cell 19 (Step 3.3)
df_nomalized.loc[:, 'bio'] = df_nomalized['bio'].apply(clean_bio)

In [47]:
df_nomalized

,userName,fullName,bio,location,externalUrl,pictureURL
0,i3mawi,adeeb amawi,,,,
1,wolterskluwerespaa,wolters kluwer espaa,,,,
2,aalishanmatrix,aalishan matrix,"computer scientist, aspiring entrepreneur & in...",,['http://www.siliconindia.com/profiles/aalisha...,
3,aaronbird,aaron bird,"co-founder & full stack ceo , the b2b marketin...","Queen Anne, Seattle, Cascadia",bizible.com,https://pbs.twimg.com/profile_images/378800000...
4,acfoto,acfoto,,,http://www.aaroncodling.com,https://scontent-ord1-1.cdninstagram.com/t51.2...
...,...,...,...,...,...,...
36802,zuzugraphics,mind in your brand,"graphic designer in san francisco - wings, hea...",,http://www.diego-t.com/index4.html,https://scontent-ord1-1.cdninstagram.com/t51.2...
36803,zuzugraphics,diego taborda,"i am a digital artist, freelance web/graphic d...",San Francisco,about.me/zuzugraphics,https://pbs.twimg.com/profile_images/857844253...
36804,tomruneberg,tom rune berg,everything is relative ;) nith skedsmo videreg...,,"['http://picasaweb.google.com/zwirwel', 'http:...",
36805,zwirwel,tom berg,,,,https://scontent-ord1-1.cdninstagram.com/t51.2...


In [49]:


df_nomalized['bio_urls'] = df_already['bio'].apply(
    lambda x: ' | '.join(clean_single_url(u) for u in extract_urls(safe_str(x))) if pd.notna(x) else ''
)
df_nomalized['bio_url_count'] = df_nomalized['bio_urls'].apply(
    lambda x: len(x.split(' | ')) if x else 0
)
df_nomalized['bio_mentions'] = df_already['bio'].apply(
    lambda x: ' | '.join(m.lower() for m in extract_mentions(safe_str(x))) if pd.notna(x) else ''
)
df_nomalized['bio_mentions_count'] = df_nomalized['bio_mentions'].apply(
    lambda x: len(x.split(' | ')) if x else 0
)

# === แสดงผล ===
comparison_df = pd.DataFrame()
comparison_df['Original_bio'] = df_already['bio']
comparison_df['bio_clean'] = df_nomalized['bio']
comparison_df['bio_urls'] = df_nomalized['bio_urls']
comparison_df['bio_url_count'] = df_nomalized['bio_url_count']
comparison_df['bio_mentions'] = df_nomalized['bio_mentions']
comparison_df['bio_mentions_count'] = df_nomalized['bio_mentions_count']
comparison_df[comparison_df['bio_clean'] != ''].head(10)


,Original_bio,bio_clean,bio_urls,bio_url_count,bio_mentions,bio_mentions_count
2,"Computer scientist, Aspiring entrepreneur & in...","computer scientist, aspiring entrepreneur & in...",,0,,0
3,"Co-Founder & Full Stack CEO @bizible, the B2B ...","co-founder & full stack ceo , the b2b marketin...",,0,bizible,1
6,Now: designer @Uber. Previously: designer @Hil...,"now: designer . previously: designer , . forev...",,0,uber | hillaryclinton | kpcbfellows,3
7,"Husband, father and illustrator | Snapchat: ic...","husband, father and illustrator snapchat: icon...",,0,iconicguy,1
8,"Me gusta estar al lado del camino, fumando el ...","me gusta estar al lado del camino, fumando el ...",,0,,0
10,"""El talento gana partidos, el trabajo en equip...","""el talento gana partidos, el trabajo en equip...",,0,,0
11,The certified twitter profile for your desired...,the certified twitter profile for your desired...,,0,,0
12,"I am a father, an avid blogger, gamer, reviewe...","i am a father, an avid blogger, gamer, reviewe...",,0,,0
13,Music industry Digital Strategist. Working rem...,music industry digital strategist. working rem...,,0,,0
14,"As cuddly as a cactus, as charming as an eel :...","as cuddly as a cactus, as charming as an eel :...",,0,,0


In [50]:
df_nomalized

,userName,fullName,bio,location,externalUrl,pictureURL,bio_urls,bio_url_count,bio_mentions,bio_mentions_count
0,i3mawi,adeeb amawi,,,,,,0,,0
1,wolterskluwerespaa,wolters kluwer espaa,,,,,,0,,0
2,aalishanmatrix,aalishan matrix,"computer scientist, aspiring entrepreneur & in...",,['http://www.siliconindia.com/profiles/aalisha...,,,0,,0
3,aaronbird,aaron bird,"co-founder & full stack ceo , the b2b marketin...","Queen Anne, Seattle, Cascadia",bizible.com,https://pbs.twimg.com/profile_images/378800000...,,0,bizible,1
4,acfoto,acfoto,,,http://www.aaroncodling.com,https://scontent-ord1-1.cdninstagram.com/t51.2...,,0,,0
...,...,...,...,...,...,...,...,...,...,...
36802,zuzugraphics,mind in your brand,"graphic designer in san francisco - wings, hea...",,http://www.diego-t.com/index4.html,https://scontent-ord1-1.cdninstagram.com/t51.2...,,0,,0
36803,zuzugraphics,diego taborda,"i am a digital artist, freelance web/graphic d...",San Francisco,about.me/zuzugraphics,https://pbs.twimg.com/profile_images/857844253...,,0,,0
36804,tomruneberg,tom rune berg,everything is relative ;) nith skedsmo videreg...,,"['http://picasaweb.google.com/zwirwel', 'http:...",,,0,,0
36805,zwirwel,tom berg,,,,https://scontent-ord1-1.cdninstagram.com/t51.2...,,0,,0


In [53]:
# === แสดงผลลัพธ์ ===
print("📊 Step 3.5: Clean Bio (Advanced) — ผลลัพธ์")
print("=" * 60)

total = len(df_nomalized)
non_empty_orig = (df_already['bio'].apply(safe_str).str.len() > 0).sum()
non_empty_clean = (df_nomalized['bio'].str.len() > 0).sum()
has_urls = (df_nomalized['bio_urls'].str.len() > 0).sum()
has_mentions = (df_nomalized['bio_mentions'].str.len() > 0).sum()

print(f"  Total profiles          : {total}")
print(f"  Non-empty bio (raw)     : {non_empty_orig} ({non_empty_orig/total*100:.1f}%)")
print(f"  Non-empty bio (clean)   : {non_empty_clean} ({non_empty_clean/total*100:.1f}%)")
print(f"  มี URL ใน bio           : {has_urls}")
print(f"  มี @mention ใน bio      : {has_mentions}")


# สถิติความยาว
bio_lengths = df_nomalized['bio'].str.len()
print(f"📏 สถิติความยาว bio_clean:")
print(f"  Mean   : {bio_lengths.mean():.1f} chars")
print(f"  Median : {bio_lengths.median():.1f} chars")
print(f"  Max    : {bio_lengths.max()} chars")
print(f"  Empty  : {(bio_lengths == 0).sum()} profiles")

print(f"\n✅ Step 3.5 เสร็จ — เพิ่ม columns: bio_clean, bio_urls, bio_url_count, bio_mentions, bio_mentions_count")


📊 Step 3.5: Clean Bio (Advanced) — ผลลัพธ์
  Total profiles          : 36807
  Non-empty bio (raw)     : 30365 (82.5%)
  Non-empty bio (clean)   : 30167 (82.0%)
  มี URL ใน bio           : 2553
  มี @mention ใน bio      : 5869
📏 สถิติความยาว bio_clean:
  Mean   : 106.6 chars
  Median : 90.0 chars
  Max    : 1875 chars
  Empty  : 6640 profiles

✅ Step 3.5 เสร็จ — เพิ่ม columns: bio_clean, bio_urls, bio_url_count, bio_mentions, bio_mentions_count


### Step 3.6: Clean Location
`location` → `location_clean`

การแปลง:
- lowercase  
- ลบ emoji  
- Normalize whitespace → trim  
- ตัวอย่าง: `  San Francisco, CA  ` → `san francisco, ca`


In [ ]:


def clean_location(text: str) -> str:
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # 1. แปลงเป็นตัวพิมพ์เล็ก
    text = text.lower()

    # 3. ลบ Emojiz
    text = remove_emojis(text)
    
    # 6. ยุบช่องว่าง
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()
    
df_nomalized.loc[:, 'location'] = df_nomalized['location'].apply(clean_location)

In [48]:
df_nomalized

,userName,fullName,bio,location,externalUrl,pictureURL,userName_clean,fullName_clean,bio_clean,bio_urls,bio_url_count,bio_mentions,bio_mentions_count
0,i3mawi,adeeb amawi,,,,,i3mawi,adeeb amawi,,,0,,0
1,wolterskluwerespaa,wolters kluwer espaa,,,,,wolterskluwerespaa,wolters kluwer espaa,,,0,,0
2,aalishanmatrix,aalishan matrix,"Computer scientist, Aspiring entrepreneur & in...",,['http://www.siliconindia.com/profiles/aalisha...,,aalishanmatrix,aalishan matrix,"computer scientist, aspiring entrepreneur & in...",,0,,0
3,aaronbird,aaron bird,"Co-Founder & Full Stack CEO @bizible, the B2B ...","queen anne, seattle, cascadia",bizible.com,https://pbs.twimg.com/profile_images/378800000...,aaronbird,aaron bird,"co-founder & full stack ceo , the b2b marketin...",,0,bizible,1
4,acfoto,acfoto,,,http://www.aaroncodling.com,https://scontent-ord1-1.cdninstagram.com/t51.2...,acfoto,acfoto,,,0,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36802,zuzugraphics,mind in your brand,"Graphic Designer in San Francisco - Wings, Hea...",,http://www.diego-t.com/index4.html,https://scontent-ord1-1.cdninstagram.com/t51.2...,zuzugraphics,mind in your brand,"graphic designer in san francisco - wings, hea...",,0,,0
36803,zuzugraphics,diego taborda,"I am a Digital Artist, freelance web/Graphic D...",san francisco,about.me/zuzugraphics,https://pbs.twimg.com/profile_images/857844253...,zuzugraphics,diego taborda,"i am a digital artist, freelance web/graphic d...",,0,,0
36804,tomruneberg,tom rune berg,Everything is relative ;) NITH Skedsmo Videreg...,,"['http://picasaweb.google.com/zwirwel', 'http:...",,tomruneberg,tom rune berg,everything is relative ;) nith skedsmo videreg...,,0,,0
36805,zwirwel,tom berg,,,,https://scontent-ord1-1.cdninstagram.com/t51.2...,zwirwel,tom berg,,,0,,0


In [57]:

# comparison ต้องอ่านจาก df ตัวเดียวกัน!
comparison_df = pd.DataFrame()
comparison_df['Original_location'] = df_already['location']
comparison_df['Cleaned_location'] = df_nomalized['location']
comparison_df.head(50)


,Original_location,Cleaned_location
0,,
1,,
2,,
3,"Queen Anne, Seattle, Cascadia","queen anne, seattle, cascadia"
4,,
5,,
6,"San Francisco, CA","san francisco, ca"
7,,
8,,
9,,


In [63]:
# ใช้ Regex '\d' เพื่อหาตัวเลข (digit) ตัวใดก็ได้ 0-9
df_with_numbers = df_already[df_already['location'].str.contains(r'\d', na=False)]

# แสดงผลออกมาดู
df_with_numbers[['userName', 'location']].head(40)
# df_with_numbers[['userName', 'location']].count()

,userName,location
15,@cityofthedes,"iPhone: 36.187325,-86.746086"
41,@aldrinshootsraw,"14.597589,120.986844"
84,@andrianinp,+62
85,@artepilpilean,World 2.0 - Bx - Bio
110,@translaticlab,Solo / 081 825 2500
184,@bpmccartney,"26.397542,-81.796433"
230,@CassWorldNYC,"ÜT: 40.709425,-74.002985"
252,@chiefie,"iPhone: -43.536060,172.569702"
257,@chrisahinojosa,"iPhone: 28.030743,-81.919441"
272,@CkAssociati,"potenza, via sicilia 67"


### Step 3.7: Clean External URL & Extract Domain
`externalUrl` → `externalUrl_clean` → `external_domain`

**externalUrl_clean** — การแปลง:
- trim → lowercase  
- ลบ protocol (`http://`, `https://`)  
- ลบ `www.`  
- ลบ query string (`?utm=1`)  
- ลบ fragment (`#about`)  
- ลบ trailing `/`  
- ตัวอย่าง: `HTTPS://www.LinkedIn.com/in/Aaron/?utm=1` → `linkedin.com/in/aaron`

**external_domain** — การแปลง:
- ถอดเฉพาะ domain จาก `externalUrl_clean`  
- ตัวอย่าง: `linkedin.com/in/aaron` → `linkedin.com`


In [ ]:
def safe_str(value) -> str:
    """Convert any value to safe string"""
    if value is None:
        return ''
    if isinstance(value, float) and math.isnan(value):
        return ''
    s = str(value).strip()
    if s.lower() in {'none', 'nan', 'null'}:
        return ''
    return s

def parse_url_list(value) -> list:
    """
    แปลง externalUrl ให้เป็น list ของ URL เสมอ
    รองรับ:
    - URL เดี่ยว: 'https://abc.com'
    - list-string: "['https://a.com', 'http://b.com']"
    - list จริง: ['https://a.com', 'http://b.com']
    """
    if isinstance(value, list):
        return [safe_str(x) for x in value if safe_str(x)]

    s = safe_str(value)
    if not s:
        return []

    # กรณีเป็น string ที่เก็บ list ไว้
    if s.startswith('[') and s.endswith(']'):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [safe_str(x) for x in parsed if safe_str(x)]
        except Exception:
            pass

    # กรณี URL เดี่ยว
    return [s]

def normalize_url(url: str) -> str:
    """
    clean URL ให้เหลือรูปแบบมาตรฐาน:
    - lowercase
    - ลบ protocol
    - ลบ www.
    - ลบ trailing slash
    - ลบ fragment (#...)
    """
    u = safe_str(url).lower()
    if not u:
        return ''

    # เติม scheme ชั่วคราว ถ้าไม่มี เพื่อให้ urlparse ทำงานถูก
    if not re.match(r'^[a-z]+://', u):
        u_for_parse = 'http://' + u
    else:
        u_for_parse = u

    parsed = urlparse(u_for_parse)

    domain = parsed.netloc.lower()
    path = parsed.path.rstrip('/')
    query = parsed.query

    # ลบ www.
    domain = re.sub(r'^www\.', '', domain)

    cleaned = domain + path
    if query:
        cleaned += '?' + query

    return cleaned.strip()

def extract_domain(cleaned_url: str) -> str:
    """
    ดึง domain จาก cleaned url
    เช่น:
    linkedin.com/in/aaron -> linkedin.com
    thailand.go.th -> thailand.go.th
    """
    s = safe_str(cleaned_url)
    if not s:
        return ''
    return s.split('/')[0]

def clean_external_url_field(value):
    """
    คืนค่า 3 อย่าง:
    - cleaned_urls
    - domains
    - url_count
    """
    url_list = parse_url_list(value)
    cleaned_urls = [normalize_url(u) for u in url_list if normalize_url(u)]
    domains = [extract_domain(u) for u in cleaned_urls if extract_domain(u)]
    return cleaned_urls, domains, len(cleaned_urls)

def squash_single_or_list(items):
    """
    ถ้ามี 1 ค่า -> คืน string
    ถ้ามีหลายค่า -> คืน list
    ถ้าไม่มี -> คืน ''
    """
    if not items:
        return ''
    if len(items) == 1:
        return items[0]
    return items
    
df_nomalized.loc[:, 'ext'] = df_nomalized['ext'].apply(clean_ext)

TypeError: Invalid value '[([], [], 0) ([], [], 0)
 (['siliconindia.com/profiles/aalishan-matrix-6i4xypz8.html', 'linkedin.com/in/aalishanmatrix', 'about.me/aalishanmatrix', 'ibm.com/developerworks/mydeveloperworks/profiles/user/aalishanmatrix', 'aalishanmatrix.com', 'google.com/partners', 'ads.bingads.microsoft.com/en-us/training-accreditation-find-a-pro-directory?memberid=8f09ad57-e6b8-49dd-8158-5ffe9e9d61f0'], ['siliconindia.com', 'linkedin.com', 'about.me', 'ibm.com', 'aalishanmatrix.com', 'google.com', 'ads.bingads.microsoft.com'], 7)
 ...
 (['picasaweb.google.com/zwirwel', 'blogger.com/profile/06752881656322158294', 'google.com/reader/shared/08285046202665493130', 'zwirwel.blogspot.com', 'twitter.com/zwirwel'], ['picasaweb.google.com', 'blogger.com', 'google.com', 'zwirwel.blogspot.com', 'twitter.com'], 5)
 ([], [], 0) (['about.me/zwirwel'], ['about.me'], 1)]' for dtype 'str'

In [ ]:


df_already['externalUrl_clean_list'] = result.apply(lambda x: x[0])    # เก็บเป็น list จริง
df_already['external_domain_list'] = result.apply(lambda x: x[1])      # เก็บเป็น list จริง
df_already['url_count'] = result.apply(lambda x: x[2])

# version สำหรับแสดงผลอ่านง่าย
df_already['externalUrl_clean'] = df_already['externalUrl_clean_list'].apply(squash_single_or_list)
df_already['external_domain'] = df_already['external_domain_list'].apply(squash_single_or_list)


# ============================================================
# Summary Stats
# ============================================================
total_profiles = len(df_already)
non_empty_raw = df_already['externalUrl'].apply(lambda x: safe_str(x) != '').sum()
non_empty_clean = (df_already['url_count'] > 0).sum()
multi_url_profiles = (df_already['url_count'] > 1).sum()
single_url_profiles = (df_already['url_count'] == 1).sum()
empty_profiles = (df_already['url_count'] == 0).sum()

url_count_series = df_already['url_count']

print("📊 Step X.X: Clean External URL — Summary")
print("=" * 60)
print(f"  Total profiles              : {total_profiles}")
print(f"  Non-empty externalUrl (raw) : {non_empty_raw} ({non_empty_raw/total_profiles*100:.1f}%)")
print(f"  Non-empty after clean       : {non_empty_clean} ({non_empty_clean/total_profiles*100:.1f}%)")
print(f"  Single URL profiles         : {single_url_profiles}")
print(f"  Multi-URL profiles          : {multi_url_profiles}")
print(f"  Empty externalUrl           : {empty_profiles}")
print()
print("📏 สถิติ url_count:")
print(f"  Mean   : {url_count_series.mean():.2f}")
print(f"  Median : {url_count_series.median():.1f}")
print(f"  Max    : {url_count_series.max()}")   # max ของ external_url
print(f"  Min    : {url_count_series.min()}")
print()

# ============================================================
# ความยาวข้อความ externalUrl ดิบ (optional)
# ============================================================
raw_len = df_already['externalUrl'].apply(lambda x: len(safe_str(x)))
print("📏 สถิติความยาว externalUrl (raw text length):")
print(f"  Mean   : {raw_len.mean():.1f} chars")
print(f"  Median : {raw_len.median():.1f} chars")
print(f"  Max    : {raw_len.max()} chars")
print(f"  Min    : {raw_len.min()} chars")
print()

# ============================================================
# ดูตัวอย่างข้อมูล
# ============================================================
preview = pd.DataFrame({
    'Original_externalUrl': df_already['externalUrl'],
    'externalUrl_clean': df_already['externalUrl_clean'],
    'external_domain': df_already['external_domain'],
    'url_count': df_already['url_count']
})

print("🔍 Sample cleaned externalUrl:")
display(preview.head())

📊 Step X.X: Clean External URL — Summary
  Total profiles              : 36807
  Non-empty externalUrl (raw) : 30580 (83.1%)
  Non-empty after clean       : 30580 (83.1%)
  Single URL profiles         : 21679
  Multi-URL profiles          : 8901
  Empty externalUrl           : 6227

📏 สถิติ url_count:
  Mean   : 2.81
  Median : 1.0
  Max    : 286
  Min    : 0

📏 สถิติความยาว externalUrl (raw text length):
  Mean   : 100.9 chars
  Median : 23.0 chars
  Max    : 13628 chars
  Min    : 0 chars

🔍 Sample cleaned externalUrl:


,Original_externalUrl,externalUrl_clean,external_domain,url_count
0,,,,0
1,,,,0
2,"['http://www.siliconindia.com/profiles/aalishan-matrix-6i4XYpZ8.html', 'http://www.linkedin.com/in/aalishanmatrix', 'http://about.me/aalishanmatrix', 'https://www.ibm.com/developerworks/mydeveloperworks/profiles/user/aalishanmatrix', 'http://www.aalishanmatrix.com', 'https://www.google.com/partners/#i_profile;idtf=118101296857513321709;', 'http://ads.bingads.microsoft.com/en-us/training-accreditation-find-a-pro-directory?MemberID=8f09ad57-e6b8-49dd-8158-5ffe9e9d61f0']","[siliconindia.com/profiles/aalishan-matrix-6i4xypz8.html, linkedin.com/in/aalishanmatrix, about.me/aalishanmatrix, ibm.com/developerworks/mydeveloperworks/profiles/user/aalishanmatrix, aalishanmatrix.com, google.com/partners, ads.bingads.microsoft.com/en-us/training-accreditation-find-a-pro-directory?memberid=8f09ad57-e6b8-49dd-8158-5ffe9e9d61f0]","[siliconindia.com, linkedin.com, about.me, ibm.com, aalishanmatrix.com, google.com, ads.bingads.microsoft.com]",7
3,bizible.com,bizible.com,bizible.com,1
4,http://www.aaroncodling.com,aaroncodling.com,aaroncodling.com,1


ขอดูข้อมูลแบบเฉพาะเจาะจง 

In [ ]:
import pandas as pd

# สั่งให้ Pandas แสดงข้อความในคอลัมน์แบบเต็ม ไม่ต้องย่อ
pd.set_option('display.max_colwidth', None)

# ลองรันการแสดงผลอีกครั้ง
row = comparison_df.iloc[1]

print(row) # แสดงทุกคอลัมน์ในแถวนั้นแบบเต็มๆ

Original_externalUrl    ['https://github.com/user', 'HTTP://WWW.EXAMPLE.COM/PAGE#TOP']
externalUrl_clean       ['https://github.com/user', 'http://www.example.com/page#top']
external_domain                                                               ['https:
url_count                                                                            2
Name: 1, dtype: object


### Step 3.8: Create Profile ID
`outputProfileName` → `profile_id`

การแปลง:
- ใช้ `outputProfileName` (ใช้ `user_folder` เป็น fallback) แล้ว normalize  
- Profile ID นี้ใช้เป็น **ground truth** สำหรับจับคู่คนเดียวกันข้ามแพลตฟอร์ม

In [ ]:
# --- Create Profile ID ---
df_clean['profile_id'] = df_clean['outputProfileName'].apply(normalize_text)

# === แสดงผลลัพธ์ ===
print("📊 Step 3.8: Create Profile ID — ผลลัพธ์")
print("=" * 60)

# สถิติ
total = len(df_clean)
unique_ids = df_clean['profile_id'].nunique()
non_empty = (df_clean['profile_id'].str.len() > 0).sum()
print(f"  Total profiles     : {total}")
print(f"  Unique profile IDs : {unique_ids}")
print(f"  Non-empty IDs      : {non_empty} ({non_empty/total*100:.1f}%)")

# ดูว่ามี profile_id ที่ซ้ำข้ามแพลตฟอร์มกี่คน (= ground truth)
cross_platform = df_clean.groupby('profile_id')['platform'].nunique()
multi_platform = cross_platform[cross_platform > 1]
print(f"  Cross-platform IDs : {len(multi_platform)} (มีข้อมูลมากกว่า 1 platform)")

# ตัวอย่าง profile_id
print(f"\n🔍 ตัวอย่าง outputProfileName → profile_id (10 rows):")
print("-" * 60)
sample = df_clean[['outputProfileName', 'profile_id', 'platform']].drop_duplicates('profile_id').head(10)
for _, row in sample.iterrows():
    print(f"  [{row['platform']:10s}] '{row['outputProfileName']}' → '{row['profile_id']}'")

# ตัวอย่าง cross-platform match
if len(multi_platform) > 0:
    print(f"\n🔗 ตัวอย่าง Cross-Platform Match (คนเดียวกันหลาย platform):")
    print("-" * 60)
    sample_ids = multi_platform.head(3).index.tolist()
    for pid in sample_ids:
        rows = df_clean[df_clean['profile_id'] == pid][['platform', 'userName', 'fullName']]
        print(f"  profile_id = '{pid}':")
        for _, r in rows.iterrows():
            print(f"    📱 {r['platform']:10s} | @{r['userName']} | {r['fullName']}")
        print()

print(f"✅ Step 3.8 เสร็จ — เพิ่ม column: profile_id")

### Step 3.9 : ทำการจัดข้อมูลให้อยู่ในสัดส่วนของตัวเอง หลังจากผ่านการ nomalize มาแล้ว 


## Stage 3 Summary
สรุปผลรวมของขั้นตอน Data Cleaning ทั้งหมด

In [ ]:
df_normalized

,externalUrl_clean,external_domain,url_count
0,linkedin.com/in/aaron?utm=1,linkedin.com,1
1,"[github.com/user, example.com/page]","[github.com, example.com]",2
2,thailand.go.th,thailand.go.th,1
3,,,0
4,,,0


In [ ]:
print("=" * 60)
print("📊 STAGE 3 SUMMARY — Data Cleaning & Standardization")
print("=" * 60)

print(f"\n  DataFrame shape: {df_clean.shape}")
print(f"  Columns เพิ่มใหม่: userName_clean, fullName_clean, bio_clean, location_clean, externalUrl_clean, profile_id")

print(f"\n  {'Column':<22s} {'Non-Empty':>10s} {'Pct':>8s}  Bar")
print(f"  {'-'*22} {'-'*10} {'-'*8}  {'-'*20}")

check_cols = [
    ('userName_clean',    'Username (norm)'),
    ('fullName_clean',    'FullName (norm)'),
    ('bio_clean',         'Bio (clean)'),
    ('location_clean',    'Location (clean)'),
    ('externalUrl_clean', 'ExtURL (clean)'),
    ('pictureURL',        'Picture URL'),
    ('profile_id',        'Profile ID'),
]

for col, label in check_cols:
    if col in df_clean.columns:
        non_empty = (df_clean[col].str.len() > 0).sum()
        total = len(df_clean)
        pct = non_empty / total * 100
        bar = chr(9608) * int(pct / 5) + chr(9617) * (20 - int(pct / 5))
        print(f"  {label:<22s} {non_empty:>10,}  {pct:>6.1f}%  {bar}")

print(f"\n  Per-Platform Distribution:")
for platform in df_clean['platform'].unique():
    count = (df_clean['platform'] == platform).sum()
    print(f"    📱 {platform:12s}: {count:>6,} profiles")

print(f"\n{'='*60}")
print(f"✅ Stage 3 COMPLETE — df_clean พร้อมใช้งานใน Stage ถัดไป")
print(f"{'='*60}")